## Bayesian Experiment Uplift (PyMC)

**Goal of this notebook**

This notebook estimates causal lift from a marketing / product experiment.

We assume we ran a test (e.g. new promo, new placement, new algorithm)
and randomly assigned customers into:
- `treatment = 1` (exposed)
- `treatment = 0` (control / holdout)

For each customer we track whether they converted (made a purchase).

We will:
1. Load the experiment table (`ab_experiment.parquet`) generated in the ETL notebook.
2. Fit a Bayesian logistic model to estimate the effect of treatment on conversion.
3. Convert posterior samples into stakeholder-facing metrics:
   - baseline conversion rate (control)
   - treatment conversion rate
   - absolute uplift
   - 95% credible interval on uplift
   - probability that uplift > 0
4. Save a clean JSON summary (`ab_summary.json`) in the outputs directory, which a future API can consume.

In [15]:
from pathlib import Path

CWD = Path().resolve()
if CWD.name in ["notebooks", "etl"]:
    PROJECT_ROOT = CWD.parent
else:
    PROJECT_ROOT = CWD

print("CWD:", CWD)
print("PROJECT_ROOT:", PROJECT_ROOT)

# NOTE: in *your* run so far, the parquet files were saved under etl/data/outputs,
# not data/outputs at the root. We will read from there.
OUTPUT_DIR = PROJECT_ROOT / "etl" / "data" / "outputs"
print("OUTPUT_DIR:", OUTPUT_DIR)
print("Exists:", OUTPUT_DIR.exists())

CWD: C:\Users\91957\Desktop\MS Admissions\Concordia University\MachineLearningProjects\RetailMarkovChain\retail-measurement-lab\notebooks
PROJECT_ROOT: C:\Users\91957\Desktop\MS Admissions\Concordia University\MachineLearningProjects\RetailMarkovChain\retail-measurement-lab
OUTPUT_DIR: C:\Users\91957\Desktop\MS Admissions\Concordia University\MachineLearningProjects\RetailMarkovChain\retail-measurement-lab\etl\data\outputs
Exists: True


### Load experiment data

The ETL notebook created a table called `ab_experiment.parquet`.

Each row represents one unique customer and includes:
- `customer_id`
- `treatment` (0 = control, 1 = exposed)
- `converted` (0/1, did they ever purchase?)

This simulates a typical growth experiment:
- Show 50% of users a new promo/experience
- See if they buy more

We'll load that table now.

In [16]:
import pandas as pd

ab_path = OUTPUT_DIR / "ab_experiment.parquet"

# Load data
ab_df = pd.read_parquet(ab_path)

# Make sure data types are numeric for modeling
ab_df["treatment"] = ab_df["treatment"].astype(int)
ab_df["converted"] = ab_df["converted"].astype(int)

print("Shape:", ab_df.shape)
print(ab_df.head())

print("\nTreatment assignment counts:")
print(ab_df["treatment"].value_counts())

print("\nOverall conversion rate:")
print(ab_df["converted"].mean())

Shape: (1407580, 3)
  customer_id  treatment  converted
0      257597          1          0
1      992329          1          0
2      111016          1          0
3      483717          1          0
4      951259          1          0

Treatment assignment counts:
treatment
1    704340
0    703240
Name: count, dtype: int64

Overall conversion rate:
0.00832563690873698


### Collapse to group-level counts for efficient modeling

The good news: for a simple A/B test with one binary treatment, we don't need
to model each individual user. All we really need is:

- How many users were in control, and how many of them converted.
- How many users were in treatment, and how many of them converted.

We compute:
- `n0`, `y0` = size of control group and number of converters in control
- `n1`, `y1` = size of treatment group and number of converters in treatment

This is statistically equivalent to using all rows for estimating overall lift,
but it's dramatically lighter and runs fast.

In [17]:
agg = (
    ab_df
    .groupby("treatment")["converted"]
    .agg(["sum", "count"])
    .rename(columns={"sum": "conversions", "count": "n"})
)

y0 = int(agg.loc[0, "conversions"])  # converters in control
n0 = int(agg.loc[0, "n"])            # users in control
y1 = int(agg.loc[1, "conversions"])  # converters in treatment
n1 = int(agg.loc[1, "n"])            # users in treatment

print("Control:", y0, "/", n0)
print("Treatment:", y1, "/", n1)

Control: 5869 / 703240
Treatment: 5850 / 704340


## Model definition (Bayesian logistic regression)

We model conversion as a Bernoulli outcome with log-odds that depend on treatment:

$$
\text{logit}(p_i) = \alpha + \beta \cdot \text{treatment}_i
$$

Parameter meanings:

$$
\alpha = \text{baseline log-odds of conversion for control (treatment}_i = 0)
$$

$$
\beta = \text{lift in log-odds from treatment (treatment}_i = 1)
$$

From $\alpha$ and $\beta$, we recover actual conversion probabilities (not just log-odds):

- Control group's conversion probability:
  
  $$
  p_{\text{control}} = \frac{1}{1 + \exp(-\alpha)}
  $$

- Treatment group's conversion probability:
  
  $$
  p_{\text{treat}} = \frac{1}{1 + \exp\left[-(\alpha + \beta)\right]}
  $$

- Absolute uplift (in percentage points of conversion rate):
  
  $$
  \text{uplift} = p_{\text{treat}} - p_{\text{control}}
  $$

- Probability the treatment actually helps:
  
  $$
  \Pr(\text{uplift} > 0)
  $$

---

### Why Bayesian?

Instead of giving only a p-value, the Bayesian model gives us full posterior distributions for all quantities above.

In [18]:
import pymc as pm
import pandas as pd
import numpy as np
import arviz as az
import json

with pm.Model() as model_binom:
    alpha = pm.Normal("alpha", mu=0, sigma=5)     # baseline log-odds
    beta  = pm.Normal("beta",  mu=0, sigma=5)     # treatment lift (log-odds)

    p_control = pm.Deterministic("p_control", pm.math.sigmoid(alpha))
    p_treat   = pm.Deterministic("p_treat",   pm.math.sigmoid(alpha + beta))

    pm.Binomial("obs_control", n=n0, p=p_control, observed=y0)
    pm.Binomial("obs_treat",   n=n1, p=p_treat,   observed=y1)

    trace = pm.sample(
        draws=2000, tune=1000, chains=2, cores=2, target_accept=0.9,
        return_inferencedata=True
    )

print(az.summary(trace, var_names=["alpha", "beta", "p_control", "p_treat"]))

Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (2 chains in 2 jobs)
NUTS: [alpha, beta]


Output()

Sampling 2 chains for 1_000 tune and 2_000 draw iterations (2_000 + 4_000 draws total) took 21 seconds.
We recommend running at least 4 chains for robust computation of convergence diagnostics


            mean     sd  hdi_3%  hdi_97%  mcse_mean  mcse_sd  ess_bulk  \
alpha     -4.777  0.013  -4.803   -4.753      0.000      0.0    1065.0   
beta      -0.005  0.019  -0.040    0.030      0.001      0.0    1165.0   
p_control  0.008  0.000   0.008    0.009      0.000      0.0    1065.0   
p_treat    0.008  0.000   0.008    0.009      0.000      0.0    3201.0   

           ess_tail  r_hat  
alpha        1518.0    1.0  
beta         1631.0    1.0  
p_control    1518.0    1.0  
p_treat      2957.0    1.0  


### Posterior-derived business metrics

Given posterior samples of the model parameters $\alpha$ and $\beta$, we compute a set of derived quantities that are directly interpretable in business terms.

Let
$$
p_{\text{control}} = \sigma(\alpha)
\quad \text{and} \quad
p_{\text{treat}} = \sigma(\alpha + \beta),
$$
where $\sigma(z) = \frac{1}{1 + e^{-z}}$ is the logistic (sigmoid) function that maps log-odds to a probability in $[0, 1]$.

These correspond to:
- $p_{\text{control}}$: baseline purchase probability (conversion rate) for the control group.
- $p_{\text{treat}}$: purchase probability for the treatment group.

We define the absolute uplift as
$$
\Delta = p_{\text{treat}} - p_{\text{control}},
$$
i.e. the absolute difference in conversion rate between treatment and control, measured in probability points.

From the posterior draws of $\alpha$ and $\beta$, we obtain posterior draws of $p_{\text{control}}$, $p_{\text{treat}}$, and $\Delta$. We summarize each of these quantities using:

1. The posterior mean (point estimate).
2. The central 95% credible interval (2.5th to 97.5th percentiles of the posterior).
3. For uplift, the posterior probability of a true positive effect:
$$
\Pr(\Delta > 0).
$$

We then collect these summaries into a dictionary (`result`) that can be serialized to JSON. This object serves as the final experiment readout and can be consumed directly by downstream reporting layers (dashboards, alerts, decision reviews).

In [19]:
import numpy as np
import json

alpha_draws = trace.posterior["alpha"].values.reshape(-1)
beta_draws  = trace.posterior["beta"].values.reshape(-1)

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

p_control_draws = sigmoid(alpha_draws)
p_treat_draws   = sigmoid(alpha_draws + beta_draws)
uplift_draws    = p_treat_draws - p_control_draws  # absolute lift in probability

def summarize(x):
    return {
        "mean": float(np.mean(x)),
        "ci_95_low": float(np.percentile(x, 2.5)),
        "ci_95_high": float(np.percentile(x, 97.5)),
    }

summary_control = summarize(p_control_draws)
summary_treat   = summarize(p_treat_draws)
summary_uplift  = summarize(uplift_draws)
prob_uplift_gt_0 = float(np.mean(uplift_draws > 0))

result = {
    "control_conversion": summary_control,
    "treatment_conversion": summary_treat,
    "absolute_uplift": summary_uplift,
    "prob_uplift_gt_0": prob_uplift_gt_0,
}

print(json.dumps(result, indent=2))

{
  "control_conversion": {
    "mean": 0.00834777779453012,
    "ci_95_low": 0.008134239658378568,
    "ci_95_high": 0.008566324231954515
  },
  "treatment_conversion": {
    "mean": 0.008305510142293698,
    "ci_95_low": 0.008085361558730988,
    "ci_95_high": 0.008519849978757387
  },
  "absolute_uplift": {
    "mean": -4.226765223642094e-05,
    "ci_95_low": -0.0003458623586993463,
    "ci_95_high": 0.00026294004788864186
  },
  "prob_uplift_gt_0": 0.38525
}


### Export experiment summary to JSON (`ab_summary.json`)

Finally, we save the summary dictionary as `ab_summary.json`
in the same `OUTPUT_DIR` as the other generated assets.

This gives us a clean, lightweight artifact that can be:
- read by a dashboard,
- sent to an API,
- or dropped directly into a report.

The JSON contains:
- control conversion rate (mean + 95% interval)
- treatment conversion rate (mean + 95% interval)
- absolute uplift (mean + 95% interval)
- probability that uplift > 0

This is the final output of the notebook.

In [21]:
summary_path = OUTPUT_DIR / "ab_summary.json"

with open(summary_path, "w") as f:
    json.dump(result, f, indent=2)

print(f"Wrote summary to {summary_path}")

Wrote summary to C:\Users\91957\Desktop\MS Admissions\Concordia University\MachineLearningProjects\RetailMarkovChain\retail-measurement-lab\etl\data\outputs\ab_summary.json
